# EVE-AR-95M — Cloud GPU training (Kaggle)

هذا الدفتر يجهّز بيئة GPU، ينزّل مجموعات البيانات العربية ذات المصادر المثبتة، يبني tokenizer من الصفر، ثم يبدأ pretraining بأوزان عشوائية. **لا تستخدمه على CPU**. التدريب من الصفر تجريبي، ولا يعني أن النموذج سيبلغ مستوى النماذج الكبيرة.

في Kaggle اختر `Settings → Accelerator → GPU T4 x2`، وفعّل Internet. الجلسات والحصة محدودة؛ قد لا تنتهي 20,000 خطوة في جلسة واحدة. تُحفظ checkpoints دورياً داخل `/kaggle/working`، لكن احفظ/نزّل المخرجات من Kaggle بعد انتهاء الجلسة.


In [ ]:
import os, sys, subprocess, json, torch
assert torch.cuda.is_available(), 'GPU is not enabled. In Kaggle: Settings → Accelerator → GPU T4 x2, then restart the session.'
print('CUDA:', torch.version.cuda)
print('GPU 0:', torch.cuda.get_device_name(0))
print('Visible GPUs:', torch.cuda.device_count())


In [ ]:
REPO = 'https://github.com/khyralslh44-hub/WALLE-Offline-New.git'
BRANCH = 'eve-companion-core-smoothness'
ROOT = '/kaggle/working/WALLE-Offline-New'
if not os.path.isdir(ROOT):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, ROOT], check=True)
os.chdir(ROOT)
print('Project:', ROOT)


In [ ]:
# Kaggle already supplies PyTorch + CUDA; install the remaining training dependencies only.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45,<5', 'tokenizers>=0.20', 'sentencepiece>=0.2', 'safetensors>=0.4', 'tqdm>=4.66', 'datasets>=3.0', 'huggingface_hub'], check=True)


In [ ]:
# Download the pinned Arabic language corpus and CIDAR instruction data.
DATA = '/kaggle/working/eve-data'
subprocess.run([sys.executable, 'eve-model/scripts/download_open_datasets.py', '--output-dir', DATA], check=True)
print(open(DATA + '/download-manifest.json', encoding='utf-8').read())


In [ ]:
# Train EVE's own byte-level BPE tokenizer from the downloaded Arabic corpus.
TOKENIZER = '/kaggle/working/eve-tokenizer'
subprocess.run([sys.executable, 'eve-model/scripts/train_tokenizer.py', '--corpus-dir', DATA + '/pretrain', '--output-dir', TOKENIZER, '--vocab-size', '24576'], check=True)


In [ ]:
# From-scratch base training. The code initializes random weights and refuses CPU by default.
# Reduce MAX_STEPS for a short smoke run; 20,000 is the current project target, not a quality guarantee.
MAX_STEPS = 20000
SAVE_EVERY = 1000
OUT = '/kaggle/working/eve-training-output/pretrain'
cmd = [sys.executable, 'eve-model/scripts/train_eve.py', '--stage', 'pretrain', '--tokenizer-dir', TOKENIZER, '--data-dir', DATA + '/pretrain', '--output-dir', OUT, '--seq-len', '1024', '--max-steps', str(MAX_STEPS), '--batch-size', '1', '--gradient-accumulation', '16', '--learning-rate', '0.0003', '--save-every', str(SAVE_EVERY)]
print('Starting:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## After pretraining

لا تدرّب شخصية إيف على ملف البذور ذي 12 مثالاً فقط؛ هذا قد يسبب الحفظ والتكرار. نحتاج أولاً إلى مجموعة شخصية أكبر ومتنوعة ومراجعة، ثم SFT وقياس جودة منفصل. بعد نجاح التدريب، تُحفظ الأوزان في `eve-training-output/pretrain/final`، ويجب تصديرها إلى GGUF واختبارها على Android قبل أي ادعاء بأنها تعمل دون إنترنت.
